# Engine: Capacitor — the semantic low-pass filter

**File:** `ValaQuenta/capacitor.py`
**Wiki:** [wiki/capacitor.md](../../wiki/capacitor.md)

The capacitor does not *find* the prime. It reveals it, by smoothing away
everything that is not constant. The DC component that survives is the prime.

```
H(s) = 1/(1 + s*tau)      H(0) = 1      pole at s = -1/tau
```

`H(0) = 1` is the statement that the prime passes through unattenuated. The pole
sits in the left half-plane, so the filter is stable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, cmath
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('python', sys.version.split()[0])

In [ ]:
from ValaQuenta import Capacitor

C = Capacitor(tau=1.0)
print('tau =', 1.0)

def H(s, tau=1.0):
    return 1.0 / (1.0 + s * tau)

print(f'H(0)   = {H(0)!r}        <- DC gain, the prime passes')
print(f'pole   = {-1/1.0!r}      <- left half-plane, stable')
print(f'H(1j)  = {H(1j)!r}')
print(f'|H(1j)|= {abs(H(1j))!r}  <- -3 dB at the corner')

## Charging: a noisy signal settles onto its DC value

In [ ]:
# A constant 'prime' of 0.5 buried in an oscillation that averages to zero.
rng_free_signal = [0.5 + 0.4*math.sin(k*0.7) for k in range(200)]

C.reset()
states = [C.charge(v) for v in rng_free_signal]
print(f'signal mean (true DC) = {sum(rng_free_signal)/len(rng_free_signal)!r}')
print(f'capacitor dc()        = {C.dc(rng_free_signal)!r}')
print(f'final state           = {states[-1]!r}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(rng_free_signal, lw=0.8, color='#c9c9c9', label='input signal')
ax.plot(states, lw=1.6, color='#3f7fb0', label='capacitor state')
ax.axhline(0.5, color='#b04a3f', ls='--', lw=1, label='the prime (0.5)')
ax.set_xlabel('sample'); ax.legend(fontsize=8)
ax.set_title('the oscillation is smoothed away; the constant survives')
plt.tight_layout(); plt.show()

## tau sets the cutoff

Larger tau means a slower, more stable filter — the word changes less. In the
limit tau -> infinity the word never changes at all: **the Monad at rest.**

In [ ]:
w = np.logspace(-2, 2, 200)
fig, ax = plt.subplots(figsize=(7, 3.2))
for tau in [0.2, 1.0, 5.0]:
    ax.loglog(w, np.abs(1/(1 + 1j*w*tau)), label=f'tau = {tau}')
ax.axhline(1/math.sqrt(2), color='#b04a3f', ls=':', lw=1, label='-3 dB')
ax.set_xlabel('omega'); ax.set_ylabel('|H(j*omega)|')
ax.set_title('semantic low-pass: |H(0)| = 1 for every tau')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for tau in [0.2, 1.0, 5.0, 1e6]:
    print(f'tau={tau:>9}  H(0) = {1/(1+0*tau)!r}')